<a href="https://colab.research.google.com/github/alextyner-tailwater/Tailwater/blob/main/Tutorials/Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In this tutorial we show how to run a basic API call to the Tailwater cloud and generate a band structure



In [ ]:
#Load Packages
!pip install tailwater
!pip install pybinding-dev
!pip install seekpath
!pip install mp_api

In [ ]:
#Define API Keys:
#Materials Project API Key
API_KEY = #Your Materials Project API Key
#Tailwater User + Password
TW_USER= # Your Tailwater Username
TW_PASS= #Your Tailwater Password

We wil begin by considering a system that has been of recent experimental interest KTaO3 (Nat Commun 14, 951 (2023))

In [ ]:
import os
from mp_api.client import MPRester
mp_id='mp-5777' #Materials Project ID for KTaO3
with MPRester(API_KEY) as mpr:
    print(f"Querying the Materials Project for {mp_id}...")


    # 1. Fetch summary data (Structure and Band Gap)
    # Passing the ID in a list to material_ids isolates the specific material
    summary_docs = mpr.materials.summary.search(
        material_ids=[mp_id],
        fields=["material_id", "structure", "band_gap","dos"]
    )

    if not summary_docs:
        print(f"Could not find summary data for {mp_id}. Check the ID.")
        #return

    doc = summary_docs[0]
    band_gap = doc.band_gap
    structure = doc.structure

In [ ]:
#Load Tailwater frontend
import numpy as np
from tailwater import (
    tw_api_call, compute_band_edges, align_to_vbm, tb_model,
    BulkDOS,
    SurfaceSpectralDensity,
    SurfaceGreensFunction,
    FermiArcMap, bulk_band_structure
)

When we call to the API we will fix "project=True" in order to extrac the necessary files for projecting into a low-energy subspace

In [ ]:

paths = tw_api_call(structure, TW_USER, TW_PASS, "./outputs", "my_mat", project=True)

In [ ]:
# Load the HDF5 the API produced — returns a tbmodels.Model with .to_pb()
model = tb_model.load("outputs/wannier90_hr.hdf5")
#Option to align VBM to zero energy for non-metals following Mat. Proj. convention
#model     = align_to_vbm(model)

In [ ]:
#Plot band structure using SeeKPath
fig = bulk_band_structure(model, auto=True, structure=structure,
                          spacing=0.02, e_range=(-8, 8))
fig

In [ ]:
#Isolate valence bands in the vicinty of the Fermi energy
#Note that energy range is defined for model prior to alignment to VBM
from tailwater import subspace_projection
subspace_projection(
    start_lr          = 1e-2,
    end_lr            = 1e-5,
    num_epochs        = 200,
    energy_range      = (-7.0, 0),       # eV, relative to E_F
    decay_sigma       = 4.0,
    device            = "cpu",
    save_path         = "./projection_out",
    embed_path        = paths["embeddings"],
    graph_output_path = paths["graph_output"],
    loss_mode         = "subspace",         # default
)

In [ ]:
#Load reduced Model and plot band structure
model = tb_model.load("projection_out/embeddings_pred.hdf5")
model     = align_to_vbm(model)
fig = bulk_band_structure(model, auto=True, structure=structure,
                          spacing=0.02, e_range=(-8, 8))
fig